# Lab 7.12 &mdash; Capstone: The Email-Drafting Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Chain the pipeline: extract, route, gather, draft (real Groq), validate
- Never auto-send -- every result is a needs_approval draft
- Run it over a SUITE of client emails and read the real drafts

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Now build it — Module 7 labs](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-07-12"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Capstone: the **email-drafting agent** (the client's Lab 4.1), end to end. It **extracts** the
query's fields, **routes** it to a team, **gathers** the order via the **gather-only tool** the Lab
7.11 agent is built from (`send_email` still withheld), **drafts** a grounded reply with the **real
Groq model**, **validates** it, and returns a **`needs_approval`** draft &mdash; it **never
auto-sends**. You run it over a **suite of client emails** and read the real drafts. The helpers below
are the ones you built through the module; you assemble them into `process`.

In [ ]:
from langchain_core.prompts import PromptTemplate

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent

# --- The pipeline pieces you built this module (provided so you can assemble the whole agent) ---
def extract(email):
    text = email.lower()
    digits = "".join(ch for ch in email if ch.isdigit())
    order_id = digits if digits else None
    if "refund" in text: intent = "refund"
    elif any(w in text for w in ("deliver", "arrive", "late", "where is")): intent = "delivery"
    elif "cancel" in text: intent = "cancel"
    else: intent = "other"
    sentiment = "negative" if any(w in text for w in ("frustrated", "angry", "late", "still")) else "neutral"
    return {"order_id": order_id, "intent": intent, "sentiment": sentiment}

TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}
def route(rec):
    return {"team": TEAMS.get(rec["intent"], "general"),
            "escalate": rec["sentiment"] == "negative" or rec["intent"] not in TEAMS}

DRAFT_PROMPT = PromptTemplate.from_template(
    "You are a customer-support agent. Using ONLY these order facts, write a short, polite reply.\n"
    "Invent no date or fact that is not given below.\n"
    "Customer name: {name}\nOrder id: {id}\nStatus: {status}\nETA: {eta}\nReply:")
def draft(order):
    # the DRAFT step calls the REAL Groq model, grounded in the order
    return with_backoff(lambda: llm.invoke(DRAFT_PROMPT.format(
        name=order["name"], id=order["id"], status=order["status"], eta=order["eta"])).content)
def validate(reply, order):
    return order["eta"] in reply

# --- The gather-only agent you assembled in Lab 7.11 (send_email WITHHELD) -- reused here ---
@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})
@tool
def get_template(kind: str) -> str:
    """Fetch a reply template by kind."""
    return TEMPLATES.get(kind, "")
@tool
def send_email(to: str, body: str) -> str:
    """Send an email. (Defined to show the capability -- but DELIBERATELY WITHHELD from the agent.)"""
    return "SENT"
def gather_tools():
    return [lookup_order, get_template]                 # gather-only -- send_email is NOT bound
def make_email_agent():
    return create_agent(llm, gather_tools())
print("helpers ready: extract, route, draft (real model), validate + the gather-only agent")

## Build it
Assemble `process`: chain the pipeline, gather via the agent's tool, draft with the **real model**, and
never send &mdash; every result is `needs_approval` or `needs_fix`.

In [ ]:
def process(email):
    rec    = extract(email)
    routed = route(rec)
    # gather via the SAME tool the Lab 7.11 agent is built from (reuse, not re-implement)
    found  = lookup_order.invoke(rec["order_id"]) if rec["order_id"] else {}
    order  = found if found.get("id") else {"id": rec["order_id"], "name": "there", "status": "unknown", "eta": "soon"}
    reply  = ___   # TODO: draft a grounded reply for this order with the REAL model (draft(order))
    ok     = validate(reply, order)
    # never auto-send: a valid draft awaits approval; an invalid one needs a fix
    status = ___   # TODO: "needs_approval" if ok else "needs_fix"
    return {"team": routed["team"], "escalate": routed["escalate"],
            "draft": reply, "status": status}

SUITE = [
    "Where is my order 4471? It's late.",
    "Please refund order 5090",
    "I want to cancel order 4471, I'm so frustrated",
]

try:
    # Structure check (no model call): the capstone reuses the Lab 7.11 gather-only agent.
    print("agent type      :", type(make_email_agent()).__name__)
    print("still withholds send_email:", "send_email" not in [t.name for t in gather_tools()])
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the whole pipeline over a suite of client emails. Each `draft` is written by the real Groq model, grounded in the gathered order; nothing is ever sent.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        for email in SUITE:
            r = process(email)
            print("EMAIL:", email)
            print("  team:", r["team"], "| escalate:", r["escalate"], "| status:", r["status"])
            print("  draft:", r["draft"][:140].replace(chr(10), " "))
            print()
        print("Every result is needs_approval / needs_fix -- the agent NEVER auto-sends.")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Each row runs the **real pipeline**: keyword extract &rarr; route &rarr; gather (real order) &rarr; **real Groq draft** &rarr; validate.
- Every `status` is `needs_approval` or `needs_fix` &mdash; **never** `sent`. The agent has no send tool; a human decides.
- Read the drafts: for a known order they name the real ETA (grounded); for an unknown order the model can only work with what it was given.

## Your turn (open task &mdash; no grader)
Add an awkward email the keyword extractor mis-reads (e.g. *"my package never showed up and I'm furious"*,
which has no id and no delivery keyword) and re-run. **What good looks like:** you can see exactly where the
*rule-based* extract fails the model &mdash; and you can argue for swapping in an LLM-based extractor (with a
closed-schema validator) as the fix. That's the honest edge of the system, and the bridge to Module 8.

---
### Key takeaway
You built the email-drafting agent end to end -- extract, route, gather, draft (real model), validate -- and it never sends on its own. That's an agent that does a job you'd pay for. Next: Module 8 orchestrates a team of them.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>